In [19]:
# Импортируем необходимые библиотеки

import pandas as pd
import numpy as np

from sklearn.cluster import DBSCAN, AgglomerativeClustering

#%pylab inline

In [20]:
# Загрузим данные

anomaly_df = pd.read_csv("data/anomaly_sample.csv")
anomaly_df.head()

,timestamp,value,is_anomaly,predicted
0,1425008573,42,False,44.072500
1,1425008873,41,False,50.709390
2,1425009173,41,False,81.405120
3,1425009473,61,False,39.950367
4,1425009773,44,False,35.350160


In [21]:
# Посмотрим на данные
anomaly_df.describe()

,timestamp,value,predicted
count,1.583000e+04,15830.000000,15830.000000
mean,1.427383e+09,85.572205,71.870715
std,1.370962e+06,321.760918,92.450520
min,1.425009e+09,0.000000,-281.389070
25%,1.426196e+09,29.000000,32.919171
50%,1.427383e+09,47.000000,49.771124
75%,1.428570e+09,76.000000,75.948052
max,1.429757e+09,13479.000000,2716.127200


# Поищем аномалии, с помощью DBSCAN

In [22]:
dbscan = DBSCAN(eps=1)

In [23]:
anomaly_df['prediction_dbscan_1'] = dbscan.fit_predict(anomaly_df['value'].values.reshape(-1, 1))
anomaly_df.head()

,timestamp,value,is_anomaly,predicted,prediction_dbscan_1
0,1425008573,42,False,44.072500,0
1,1425008873,41,False,50.709390,0
2,1425009173,41,False,81.405120,0
3,1425009473,61,False,39.950367,0
4,1425009773,44,False,35.350160,0


In [24]:
# Обнаруженные аномалии
anomaly_df[anomaly_df.prediction_dbscan_1 < 0]

,timestamp,value,is_anomaly,predicted,prediction_dbscan_1
162,1425057173,456,True,89.710290,-1
163,1425057473,440,True,134.684600,-1
164,1425057773,477,True,126.210050,-1
1006,1425310373,346,True,68.731980,-1
1168,1425358973,446,True,71.947266,-1
...,...,...,...,...,...
15464,1429647773,2510,True,407.389860,-1
15465,1429648073,1299,True,388.840450,-1
15466,1429648373,714,False,456.416630,-1
15467,1429648673,576,False,323.319300,-1


# Поиск аномалий с помощью агломеративной кластеризации

In [25]:
agglom = AgglomerativeClustering(n_clusters=None, distance_threshold=1)

In [26]:
anomaly_df['prediction_agglom_1'] = agglom.fit_predict(anomaly_df['value'].values.reshape(-1, 1))
anomaly_df.head()

,timestamp,value,is_anomaly,predicted,prediction_dbscan_1,prediction_agglom_1
0,1425008573,42,False,44.072500,0,2
1,1425008873,41,False,50.709390,0,0
2,1425009173,41,False,81.405120,0,0
3,1425009473,61,False,39.950367,0,15
4,1425009773,44,False,35.350160,0,1


In [27]:
valcount = anomaly_df['prediction_agglom_1'].value_counts()
valcount = set(valcount[valcount == 1].index)
anomaly_df[anomaly_df.prediction_agglom_1.isin(valcount)]

,timestamp,value,is_anomaly,predicted,prediction_dbscan_1,prediction_agglom_1
162,1425057173,456,True,89.710290,-1,630
164,1425057773,477,True,126.210050,-1,629
1168,1425358973,446,True,71.947266,-1,628
1360,1425416573,1698,True,101.339670,-1,607
1361,1425416873,3228,True,148.821490,-1,479
...,...,...,...,...,...,...
15465,1429648073,1299,True,388.840450,-1,462
15466,1429648373,714,False,456.416630,-1,390
15467,1429648673,576,False,323.319300,-1,396
15468,1429648973,490,False,348.968020,16,577


Задание 5.1
 
Сколько аномалий будет обнаружено при использовании DBSCAN с параметрами: число точек в кластере — 5, eps-окрестность — 2?

In [29]:
dbscan_q = DBSCAN(eps=2, min_samples=5)
labels_q = dbscan_q.fit_predict(anomaly_df['value'].values.reshape(-1, 1))
num_anomalies = (labels_q == -1).sum()
num_anomalies

np.int64(236)

Задание 5.2
 
Сколько аномалий будет обнаружено при использовании DBSCAN с параметрами: число точек в кластере — 10, eps-окрестность — 5?

In [30]:
dbscan_q2 = DBSCAN(eps=5, min_samples=10)
labels_q2 = dbscan_q2.fit_predict(anomaly_df['value'].values.reshape(-1, 1))
num_anomalies_q2 = (labels_q2 == -1).sum()
num_anomalies_q2

np.int64(226)

Задание 5.3
 
Сколько аномалий будет обнаружено при использовании агломеративной кластеризации с дистанцией отсечения 2?

In [31]:
agglom_q = AgglomerativeClustering(n_clusters=None, distance_threshold=2)
labels_agglom_q = agglom_q.fit_predict(anomaly_df['value'].values.reshape(-1, 1))
counts_agglom_q = pd.Series(labels_agglom_q).value_counts()
singletons_q = set(counts_agglom_q[counts_agglom_q == 1].index)
num_anomalies_agglom_q = int(pd.Series(labels_agglom_q).isin(singletons_q).sum())
num_anomalies_agglom_q

147